# Notebook 06 — Churn Prediction Modelling

**Objective:** Train and evaluate three baseline churn prediction models on the leakage-safe `customer_churn_dataset.csv`.

**Scope:**
- Stratified train / validation / test split (70 / 15 / 15)
- Feature preprocessing pipeline (log-transform, scale, encode)
- Logistic Regression (L2), Random Forest, XGBoost
- Evaluation: ROC-AUC, PR-AUC, F1 (churn class), confusion matrix
- Churn probability scoring on full dataset
- Risk segmentation (High / Medium / Low)
- Save scored dataset and model artifacts

**Hard rules:**
- `data/raw/` is never touched
- Revenue from `fact_invoice_items` / `dim_invoices` only
- `transactions_clean.csv` is never used
- Test set is held out until final evaluation


## 0. Imports

In [1]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay,
    roc_curve, precision_recall_curve
)
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')

# Paths
PROCESSED = '../data/processed'
FIGURES   = 'figures'
MODELS    = '../models'
os.makedirs(FIGURES, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

SEED = 42
print('Environment ready.')

Environment ready.


## 1. Load Data

In [2]:
df_raw = pd.read_csv(f'{PROCESSED}/customer_churn_dataset.csv')
print(f'Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} cols')
print(df_raw.dtypes)
df_raw.head(3)

Loaded: 5,052 rows × 11 cols
Customer_ID          int64
Order_Count          int64
Total_Revenue      float64
Avg_Order_Value    float64
First_Purchase         str
Last_Purchase          str
Recency              int64
Frequency            int64
Monetary           float64
customer_type          str
Churn_Status         int64
dtype: object


,Customer_ID,Order_Count,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase,Recency,Frequency,Monetary,customer_type,Churn_Status
0,1000,2,60.15,30.075000,2015-03-15,2015-05-27,34,2,60.15,private,0
1,1001,3,99.82,33.273333,2015-01-20,2015-05-02,59,3,99.82,private,1
2,1002,1,44.71,44.710000,2015-04-26,2015-04-26,65,1,44.71,private,0


## 2. Feature Engineering & Preprocessing

**Transformations applied:**
- Drop identifier, date strings, and alias columns (`Customer_ID`, `First_Purchase`, `Last_Purchase`, `Total_Revenue`, `Order_Count`)
- Add `tenure_days = (Last_Purchase − First_Purchase).days`
- Log1p-transform: `Monetary`, `Avg_Order_Value`, `Frequency` (all have skewness > 9)
- Binary-encode `customer_type`: `wholesaler=1`, `private=0`
- StandardScaler on all numeric features


In [3]:
def build_features(df):
    d = df.copy()
    # Tenure in obs window
    d['tenure_days'] = (
        pd.to_datetime(d['Last_Purchase']) - pd.to_datetime(d['First_Purchase'])
    ).dt.days
    # Drop identifiers, dates, and duplicate aliases
    drop_cols = ['Customer_ID', 'First_Purchase', 'Last_Purchase',
                 'Total_Revenue', 'Order_Count']  # aliases of Monetary / Frequency
    d = d.drop(columns=drop_cols)
    # Encode segment
    d['is_wholesaler'] = (d['customer_type'] == 'wholesaler').astype(int)
    d = d.drop(columns='customer_type')
    return d

df_feat = build_features(df_raw)

FEATURE_COLS = [c for c in df_feat.columns if c != 'Churn_Status']
print('Features:', FEATURE_COLS)
print(f'\nFeature matrix shape: {df_feat[FEATURE_COLS].shape}')
df_feat[FEATURE_COLS].describe().round(3)

Features: ['Avg_Order_Value', 'Recency', 'Frequency', 'Monetary', 'tenure_days', 'is_wholesaler']

Feature matrix shape: (5052, 6)


,Avg_Order_Value,Recency,Frequency,Monetary,tenure_days,is_wholesaler
count,5052.000,5052.000,5052.000,5052.000,5052.000,5052.000
mean,238.808,67.920,2.159,699.015,38.237,0.186
std,1172.370,50.376,2.954,3415.583,52.638,0.389
min,2.190,0.000,1.000,2.190,0.000,0.000
25%,30.914,23.000,1.000,43.165,0.000,0.000
50%,108.255,56.000,1.000,136.015,0.000,0.000
75%,305.820,106.000,2.000,554.275,76.000,0.000
max,77183.600,180.000,67.000,127410.230,177.000,1.000


In [4]:
# Log1p columns
LOG_COLS = ['Monetary', 'Avg_Order_Value', 'Frequency']

X = df_feat[FEATURE_COLS].copy()
y = df_feat['Churn_Status'].values

# Apply log1p before passing to pipeline (sklearn FunctionTransformer cannot select columns easily)
X[LOG_COLS] = np.log1p(X[LOG_COLS])

print('X shape:', X.shape, '| y shape:', y.shape)
print('Churn rate:', round(y.mean() * 100, 2), '%')
print('Nulls in X:', X.isnull().sum().sum())

X shape: (5052, 6) | y shape: (5052,)
Churn rate: 37.45 %
Nulls in X: 0


## 3. Train / Validation / Test Split

**Strategy:** Stratified by `Churn_Status` (preserves 37.45% churn rate in all splits).  
**Sizes:** 70% train, 15% validation, 15% test.  
The test set is **held out** and used only once at the very end.


In [5]:
# First split: train+val (85%) vs test (15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
# Second split: train (70% of total) vs val (15% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15/0.85, random_state=SEED, stratify=y_trainval
)

for name, ys in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'{name}: {len(ys):>4} rows | churn rate: {ys.mean()*100:.2f}%')

# Save split indices for reproducibility
split_meta = {
    'train_size': len(y_train),
    'val_size':   len(y_val),
    'test_size':  len(y_test),
    'seed': SEED,
    'stratified': True
}
with open(f'{PROCESSED}/split_metadata.json', 'w') as f:
    json.dump(split_meta, f, indent=2)
print('Split metadata saved.')

Train: 3536 rows | churn rate: 37.44%


Val:  758 rows | churn rate: 37.47%
Test:  758 rows | churn rate: 37.47%
Split metadata saved.


## 4. Build Pipelines

Three pipelines — each wrapping StandardScaler + classifier:

In [6]:
scale_neg_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'XGBoost scale_pos_weight: {scale_neg_weight:.3f}')

pipelines = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            C=1.0, solver='lbfgs', max_iter=1000,
            class_weight='balanced', random_state=SEED
        ))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=300, max_depth=8,
            min_samples_leaf=5,
            class_weight='balanced', random_state=SEED,
            n_jobs=-1
        ))
    ]),
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', xgb.XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_neg_weight,
            eval_metric='logloss', random_state=SEED,
            verbosity=0, use_label_encoder=False
        ))
    ])
}
print('Pipelines defined:', list(pipelines.keys()))

XGBoost scale_pos_weight: 1.671
Pipelines defined: ['Logistic Regression', 'Random Forest', 'XGBoost']


## 5. Train & Validate

In [7]:
def evaluate(pipe, X_eval, y_eval, split_label):
    proba = pipe.predict_proba(X_eval)[:, 1]
    pred  = pipe.predict(X_eval)
    return {
        'split':     split_label,
        'roc_auc':   round(roc_auc_score(y_eval, proba), 4),
        'pr_auc':    round(average_precision_score(y_eval, proba), 4),
        'f1_churn':  round(f1_score(y_eval, pred, pos_label=1), 4),
        'precision_churn': round(precision_score(y_eval, pred, pos_label=1), 4),
        'recall_churn':    round(recall_score(y_eval, pred, pos_label=1), 4),
    }

results = {}
for name, pipe in pipelines.items():
    print(f'Training: {name} ...', end=' ')
    pipe.fit(X_train, y_train)
    train_m = evaluate(pipe, X_train, y_train, 'train')
    val_m   = evaluate(pipe, X_val,   y_val,   'val')
    results[name] = {'train': train_m, 'val': val_m}
    print(f'Val ROC-AUC={val_m["roc_auc"]:.4f} | PR-AUC={val_m["pr_auc"]:.4f} | F1={val_m["f1_churn"]:.4f}')

print('\nTraining complete.')

Training: Logistic Regression ... 

Val ROC-AUC=0.6676 | PR-AUC=0.5016 | F1=0.5867
Training: Random Forest ... 

Val ROC-AUC=0.6805 | PR-AUC=0.5091 | F1=0.5967
Training: XGBoost ... 

Val ROC-AUC=0.6805 | PR-AUC=0.5034 | F1=0.5697

Training complete.


## 6. Validation Performance Summary

In [8]:
rows = []
for model, splits in results.items():
    for split, m in splits.items():
        row = {'Model': model, 'Split': split}
        row.update(m)
        rows.append(row)

df_results = pd.DataFrame(rows).set_index(['Model','Split'])
df_results = df_results.drop(columns='split')
print(df_results.to_string())

                           roc_auc  pr_auc  f1_churn  precision_churn  recall_churn
Model               Split                                                          
Logistic Regression train   0.6998  0.5294    0.6159           0.4842        0.8459
                    val     0.6676  0.5016    0.5867           0.4600        0.8099
Random Forest       train   0.8288  0.7175    0.6685           0.5230        0.9260
                    val     0.6805  0.5091    0.5967           0.4688        0.8204
XGBoost             train   0.9198  0.8603    0.7856           0.6765        0.9366
                    val     0.6805  0.5034    0.5697           0.4923        0.6761


## 7. ROC & Precision-Recall Curves (Validation Set)

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#3b82d4', '#e47b30', '#27a86a']

for (name, pipe), color in zip(pipelines.items(), colors):
    proba = pipe.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc = roc_auc_score(y_val, proba)
    axes[0].plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')

    prec, rec, _ = precision_recall_curve(y_val, proba)
    ap = average_precision_score(y_val, proba)
    axes[1].plot(rec, prec, lw=2, color=color, label=f'{name} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Validation Set')
axes[0].legend(fontsize=9)
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])

baseline_pr = y_val.mean()
axes[1].axhline(baseline_pr, color='black', linestyle='--', lw=1, label=f'Baseline ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves — Validation Set')
axes[1].legend(fontsize=9)
axes[1].set_xlim([-0.02, 1.02])
axes[1].set_ylim([-0.02, 1.02])

plt.tight_layout()
fig.savefig(f'{FIGURES}/viz6_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: viz6_roc_pr_curves.png')

Figure saved: viz6_roc_pr_curves.png


## 8. Confusion Matrices (Validation Set, threshold=0.5)

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, pipe) in zip(axes, pipelines.items()):
    pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, pred)
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        cmap='Blues', linewidths=0.5,
        xticklabels=['Retained','Churned'],
        yticklabels=['Retained','Churned']
    )
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Validation Set', fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig(f'{FIGURES}/viz7_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: viz7_confusion_matrices.png')

Figure saved: viz7_confusion_matrices.png


## 9. Feature Importances (Best Model = XGBoost)

In [11]:
best_pipe = pipelines['XGBoost']
best_clf  = best_pipe.named_steps['clf']

importances = pd.Series(
    best_clf.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ['#3b82d4' if v < importances.max() else '#e47b30' for v in importances]
ax.barh(importances.index, importances.values, color=colors_bar)
ax.set_xlabel('XGBoost Feature Importance (gain)')
ax.set_title('Feature Importances — XGBoost')
plt.tight_layout()
fig.savefig(f'{FIGURES}/viz8_feature_importances.png', dpi=150, bbox_inches='tight')
plt.close()

print('Feature importances (sorted):')
print(importances.sort_values(ascending=False).to_string())
print('Figure saved: viz8_feature_importances.png')

Feature importances (sorted):
is_wholesaler      0.784180
Monetary           0.048895
tenure_days        0.047259
Recency            0.044931
Avg_Order_Value    0.039809
Frequency          0.034926
Figure saved: viz8_feature_importances.png


## 10. Final Evaluation on Test Set (XGBoost — Best Model)

In [12]:
test_m = evaluate(best_pipe, X_test, y_test, 'test')
print('=== TEST SET RESULTS (XGBoost) ===')
for k, v in test_m.items():
    if k != 'split':
        print(f'  {k}: {v}')

print()
print(classification_report(
    y_test, best_pipe.predict(X_test),
    target_names=['Retained', 'Churned']
))

=== TEST SET RESULTS (XGBoost) ===
  roc_auc: 0.6715
  pr_auc: 0.4891
  f1_churn: 0.5826
  precision_churn: 0.4951
  recall_churn: 0.7077

              precision    recall  f1-score   support

    Retained       0.76      0.57      0.65       474
     Churned       0.50      0.71      0.58       284

    accuracy                           0.62       758
   macro avg       0.63      0.64      0.62       758
weighted avg       0.66      0.62      0.63       758



## 11. Churn Probability Scoring — Full Dataset

In [13]:
# Score all 5,052 customers with the best model
proba_all = best_pipe.predict_proba(X)[:, 1]

df_scored = df_raw[['Customer_ID', 'customer_type', 'Churn_Status',
                     'Recency', 'Frequency', 'Monetary', 'Avg_Order_Value']].copy()
df_scored['churn_probability'] = proba_all.round(4)

# Risk segmentation
def risk_segment(p):
    if p >= 0.70:
        return 'High'
    elif p >= 0.40:
        return 'Medium'
    else:
        return 'Low'

df_scored['risk_segment'] = df_scored['churn_probability'].apply(risk_segment)

print('Risk segment distribution:')
seg_dist = df_scored.groupby('risk_segment').agg(
    customers=('Customer_ID', 'count'),
    actual_churners=('Churn_Status', 'sum'),
    avg_probability=('churn_probability', 'mean'),
    total_revenue=('Monetary', 'sum')
).round(2)
seg_dist['churn_rate'] = (seg_dist['actual_churners'] / seg_dist['customers'] * 100).round(1)
seg_dist['revenue_share'] = (seg_dist['total_revenue'] / df_scored['Monetary'].sum() * 100).round(1)
print(seg_dist.to_string())

# Save scored dataset
df_scored.to_csv(f'{PROCESSED}/customer_churn_scored.csv', index=False)
print('\nScored dataset saved: data/processed/customer_churn_scored.csv')

Risk segment distribution:
              customers  actual_churners  avg_probability  total_revenue  churn_rate  revenue_share
risk_segment                                                                                       
High                943              743             0.78      330754.42        78.8            9.4
Low                1793              126             0.14     2808907.25         7.0           79.5
Medium             2316             1023             0.56      391761.80        44.2           11.1

Scored dataset saved: data/processed/customer_churn_scored.csv


## 12. Churn Probability Distribution by Segment

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: probability distribution by actual label
for label, color, name in [(0, '#27a86a', 'Retained'), (1, '#e04747', 'Churned')]:
    axes[0].hist(
        df_scored[df_scored['Churn_Status'] == label]['churn_probability'],
        bins=40, alpha=0.65, color=color, label=name, density=True
    )
axes[0].set_xlabel('Churn Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Predicted Churn Probability by Actual Label')
axes[0].legend()

# Right: probability distribution by customer type
for ctype, color in [('private', '#3b82d4'), ('wholesaler', '#7c5cd8')]:
    axes[1].hist(
        df_scored[df_scored['customer_type'] == ctype]['churn_probability'],
        bins=40, alpha=0.65, color=color, label=ctype.capitalize(), density=True
    )
axes[1].set_xlabel('Churn Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Predicted Churn Probability by Customer Type')
axes[1].legend()

plt.tight_layout()
fig.savefig(f'{FIGURES}/viz9_churn_probability_dist.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: viz9_churn_probability_dist.png')

Figure saved: viz9_churn_probability_dist.png


## 13. Revenue at Risk by Risk Segment

In [15]:
# Revenue at risk = sum of Monetary for actual churners in each segment
risk_rev = df_scored[df_scored['Churn_Status'] == 1].groupby('risk_segment').agg(
    churned_customers=('Customer_ID', 'count'),
    revenue_at_risk=('Monetary', 'sum')
).reindex(['High', 'Medium', 'Low']).fillna(0)

print('Revenue at risk (actual churners, by model risk segment):')
print(risk_rev.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
colors_risk = ['#e04747', '#e47b30', '#27a86a']
bars = ax.bar(risk_rev.index, risk_rev['revenue_at_risk'], color=colors_risk, width=0.5)
for bar, val in zip(bars, risk_rev['revenue_at_risk']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'£{val:,.0f}', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Risk Segment')
ax.set_ylabel('Revenue at Risk (£)')
ax.set_title('Revenue at Risk by Risk Segment (Actual Churners)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
fig.savefig(f'{FIGURES}/viz10_revenue_at_risk.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: viz10_revenue_at_risk.png')

Revenue at risk (actual churners, by model risk segment):
              churned_customers  revenue_at_risk
risk_segment                                    
High                        743        297313.31
Medium                     1023        194136.33
Low                         126         97108.23
Figure saved: viz10_revenue_at_risk.png


## 14. Save Model Artifacts

In [16]:
# Save all three pipelines
for name, pipe in pipelines.items():
    safe_name = name.lower().replace(' ', '_')
    path = f'{MODELS}/{safe_name}_pipeline.pkl'
    joblib.dump(pipe, path)
    print(f'Saved: {path}')

# Save model evaluation summary
summary = {}
for name, splits in results.items():
    summary[name] = splits
summary['XGBoost']['test'] = test_m

with open(f'{PROCESSED}/model_evaluation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Evaluation summary saved: data/processed/model_evaluation_summary.json')

# Save feature list used
with open(f'{PROCESSED}/model_features.json', 'w') as f:
    json.dump({'features': FEATURE_COLS, 'log_transformed': LOG_COLS}, f, indent=2)
print('Feature list saved: data/processed/model_features.json')

Saved: ../models/logistic_regression_pipeline.pkl


Saved: ../models/random_forest_pipeline.pkl
Saved: ../models/xgboost_pipeline.pkl
Evaluation summary saved: data/processed/model_evaluation_summary.json
Feature list saved: data/processed/model_features.json


## 15. Modelling Summary

### Results printed below for the report:


In [17]:
print('=' * 60)
print('MODELLING SUMMARY')
print('=' * 60)
print()
print('--- VALIDATION SET ---')
for name in pipelines:
    m = results[name]['val']
    print(f'{name:20s}  ROC-AUC={m["roc_auc"]:.4f}  PR-AUC={m["pr_auc"]:.4f}  F1={m["f1_churn"]:.4f}  Prec={m["precision_churn"]:.4f}  Rec={m["recall_churn"]:.4f}')

print()
print('--- TEST SET (XGBoost only) ---')
m = test_m
print(f'XGBoost               ROC-AUC={m["roc_auc"]:.4f}  PR-AUC={m["pr_auc"]:.4f}  F1={m["f1_churn"]:.4f}  Prec={m["precision_churn"]:.4f}  Rec={m["recall_churn"]:.4f}')

print()
print('--- RISK SEGMENT SUMMARY (all 5,052 customers) ---')
print(seg_dist.to_string())
print()
print('Notebook 06 complete.')

MODELLING SUMMARY

--- VALIDATION SET ---
Logistic Regression   ROC-AUC=0.6676  PR-AUC=0.5016  F1=0.5867  Prec=0.4600  Rec=0.8099
Random Forest         ROC-AUC=0.6805  PR-AUC=0.5091  F1=0.5967  Prec=0.4688  Rec=0.8204
XGBoost               ROC-AUC=0.6805  PR-AUC=0.5034  F1=0.5697  Prec=0.4923  Rec=0.6761

--- TEST SET (XGBoost only) ---
XGBoost               ROC-AUC=0.6715  PR-AUC=0.4891  F1=0.5826  Prec=0.4951  Rec=0.7077

--- RISK SEGMENT SUMMARY (all 5,052 customers) ---
              customers  actual_churners  avg_probability  total_revenue  churn_rate  revenue_share
risk_segment                                                                                       
High                943              743             0.78      330754.42        78.8            9.4
Low                1793              126             0.14     2808907.25         7.0           79.5
Medium             2316             1023             0.56      391761.80        44.2           11.1

Notebook 06 complete